In [ ]:
from __future__ import annotations

import pickle
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt

from experiment_config import (
    ISOFLOP_BLOCK_LEN,
    MODEL_SIZES,
    build_command,
    compute_isoflop_steps,
    dropout_for_model,
    get_optimal_lr,
)

ROOT = Path.cwd()
assert (ROOT / "train.py").exists(), "Open this notebook from the repo root so train.py is available."

DATA_ROOT = ROOT / "data"
FIG_ROOT = ROOT / "figures"
DATA_ROOT.mkdir(exist_ok=True)
FIG_ROOT.mkdir(exist_ok=True)

# ── IsoFLOP C3 config ──
ISOFLOP_LABEL = "C3"
ISOFLOP_BUDGET = 6e14
ISOFLOP_NONBLOCK_MODELS = ["remasked", "mdlm"]
ISOFLOP_BLOCK_MODELS = [
    "block_remasked",
    "block_mdlm",
    "block_edit_one_pass",
    "block_edit_two_pass",
]
ISOFLOP_NONBLOCK_SIZES = ["0.3M", "0.5M", "1M", "2M", "3M"]
SIZE_ORDER = sorted(MODEL_SIZES, key=lambda size: MODEL_SIZES[size][2])

# ── Curriculum config ──
CURRICULUM_1200_DIR = "curriculum_1200"
CURRICULUM_1200_SIZES = ["0.3M", "1M"]
CURRICULUM_1200_P_VALUES = [0.0, 0.1, 0.3, 0.5, 0.7]
CURRICULUM_1200_TOTAL_STEPS = 1200

CURRICULUM_4000_DIR = "curriculum_4000"
CURRICULUM_4000_SIZES = ["0.3M", "1M"]
CURRICULUM_4000_P_VALUES = [0.0, 0.3, 0.5]
CURRICULUM_4000_TOTAL_STEPS = 4000
CURRICULUM_4000_EVAL_INTERVAL = 50

# ── Shared training config ──
BATCH_SIZE = 128
BLOCK_SIZE = 256
BLOCK_LEN = ISOFLOP_BLOCK_LEN
EVAL_ITERS = 50
TIMEOUT_S = 7200

# ── Parallelism for A100 40GB (8 CPU cores) ──
# VRAM per process: 0.1-0.5M ~2GB, 1M ~4GB, 2-3M ~6-8GB
# CPU-bound at 8 cores, so cap there for small models.
PARALLEL_BY_SIZE = {"0.1M": 8, "0.3M": 8, "0.5M": 8, "1M": 6, "2M": 4, "3M": 4}
PARALLEL_HEAVY = 2  # for block_edit_two_pass at large sizes

# ── Plot styling ──
MODEL_LABELS = {
    "remasked": "Remasked/MDLM",
    "mdlm": "MDLM",
    "block_remasked": "Block Remasked/MDLM",
    "block_mdlm": "Block MDLM",
    "block_edit_one_pass": "Block Edit-1",
    "block_edit_two_pass": "Block Edit-2",
}

MODEL_COLORS = {
    "remasked": "#c44e52",
    "block_remasked": "#c44e52",
    "mdlm": "#4c72b0",
    "block_mdlm": "#4c72b0",
    "block_edit_one_pass": "#dd8452",
    "block_edit_two_pass": "#55a868",
}

SIZE_COLORS = {
    "0.3M": "#55a868",
    "1M": "#4c72b0",
    "3M": "#dd8452",
}

plt.rcParams.update(
    {
        "figure.dpi": 140,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 11,
    }
)


# ── Helpers ──

def format_p_value(p_value: float) -> str:
    return f"{p_value:.1f}"


def model_dir_name(model: str, block_len: int | None = None) -> str:
    if block_len is not None and model.startswith("block_"):
        return f"{model}_bl{block_len}"
    return model


def curriculum_run_dir(dataset_name: str, size: str, p_value: float) -> Path:
    return DATA_ROOT / dataset_name / f"p{format_p_value(p_value)}_{size}"


def loss_path(run_dir: Path) -> Path:
    return Path(run_dir) / "loss.pkl"


def load_loss(path_or_dir: Path):
    path = Path(path_or_dir)
    if path.is_dir():
        path = path / "loss.pkl"
    with open(path, "rb") as f:
        return pickle.load(f)


def already_done(run_dir: Path) -> bool:
    path = loss_path(run_dir)
    if not path.exists():
        return False
    try:
        log = load_loss(path)
    except Exception:
        return False
    return bool(log.get("train")) and bool(log.get("val"))


def tail(text: str, n: int = 20) -> str:
    lines = [line for line in text.splitlines() if line.strip()]
    return "\n".join(lines[-n:])


def final_metric(log, split: str = "val") -> float:
    return log[split][-1][1]


def curve_points(log, split: str = "train", offset: int = 0):
    xs = [offset + step for step, _ in log.get(split, [])]
    ys = [value for _, value in log.get(split, [])]
    return xs, ys


def isoflop_eval_interval(steps: int) -> int:
    return max(steps // 12, 20)


def isoflop_warmup_iters(steps: int) -> int:
    return min(100, max(10, steps // 20))


def curriculum_steps(total_steps: int, p_value: float) -> tuple[int, int]:
    ar_steps = int(round(total_steps * p_value))
    return ar_steps, total_steps - ar_steps


def ar_warmup_iters(ar_steps: int) -> int:
    if ar_steps <= 0:
        return 0
    return max(1, int(round(0.20 * ar_steps)))


def bd_warmup_iters(bd_steps: int) -> int:
    if bd_steps <= 0:
        return 0
    return min(50, bd_steps)


def _run_training_subprocess(cmd, cwd, timeout_s):
    """Subprocess wrapper suitable for ProcessPoolExecutor."""
    proc = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True,
                          timeout=timeout_s, check=False)
    return proc.returncode, proc.stdout, proc.stderr


def run_training(
    model: str,
    size: str,
    out_dir: Path,
    *,
    max_iters: int,
    block_len: int | None = None,
    lr: float | None = None,
    dropout: float | None = None,
    eval_interval: int = 300,
    eval_iters: int = EVAL_ITERS,
    warmup_iters: int = 100,
    resume_from: Path | None = None,
    batch_size: int = BATCH_SIZE,
    block_size: int = BLOCK_SIZE,
    save_interval: int = 0,
    gpt2_eval_interval: int = 0,
    gpt2_eval_samples: int = 0,
    sample_interval: int = 0,
    num_final_samples: int = 0,
    timeout_s: int = TIMEOUT_S,
    force: bool = False,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not force and already_done(out_dir):
        print(f"[skip] {out_dir}")
        return load_loss(out_dir)

    cmd = build_command(
        model,
        size,
        str(out_dir),
        max_iters=max_iters,
        batch_size=batch_size,
        block_size=block_size,
        block_len=block_len,
        dropout=dropout,
        lr=lr,
        eval_interval=eval_interval,
        eval_iters=eval_iters,
        warmup_iters=warmup_iters,
        gpt2_eval_interval=gpt2_eval_interval,
        gpt2_eval_samples=gpt2_eval_samples,
        save_interval=save_interval,
        num_final_samples=num_final_samples,
        sample_interval=sample_interval,
    )
    if resume_from is not None:
        cmd.extend(["--resume_from", str(resume_from)])

    print(f"[run] {model} {size} -> {out_dir}")
    proc = subprocess.run(
        cmd,
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=timeout_s,
        check=False,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"Training failed for {model} {size} in {out_dir}\n"
            f"stdout tail:\n{tail(proc.stdout)}\n\n"
            f"stderr tail:\n{tail(proc.stderr)}"
        )
    return load_loss(out_dir)


def run_parallel_batch(run_configs, max_workers=4, label="batch"):
    """Run a list of independent training configs in parallel.

    Each config is a dict with keys matching run_training() arguments.
    Returns list of (config, log) tuples.
    """
    # Filter out already-done runs
    todo = []
    results = []
    for cfg in run_configs:
        out_dir = Path(cfg["out_dir"])
        if already_done(out_dir):
            print(f"[skip] {out_dir}")
            results.append((cfg, load_loss(out_dir)))
        else:
            todo.append(cfg)

    if not todo:
        print(f"[{label}] All {len(run_configs)} runs already done")
        return results

    print(f"[{label}] Running {len(todo)} jobs (parallel={max_workers}, {len(results)} skipped)")
    t0 = time.time()

    def _make_cmd(cfg):
        cmd = build_command(
            cfg["model"], cfg["size"], str(cfg["out_dir"]),
            max_iters=cfg["max_iters"],
            batch_size=cfg.get("batch_size", BATCH_SIZE),
            block_size=cfg.get("block_size", BLOCK_SIZE),
            block_len=cfg.get("block_len"),
            dropout=cfg.get("dropout"),
            lr=cfg.get("lr"),
            eval_interval=cfg.get("eval_interval", 300),
            eval_iters=cfg.get("eval_iters", EVAL_ITERS),
            warmup_iters=cfg.get("warmup_iters", 100),
            gpt2_eval_interval=0, gpt2_eval_samples=0,
            save_interval=cfg.get("save_interval", 0),
            num_final_samples=0, sample_interval=0,
        )
        if cfg.get("resume_from"):
            cmd.extend(["--resume_from", str(cfg["resume_from"])])
        return cmd

    # Ensure output dirs exist
    for cfg in todo:
        Path(cfg["out_dir"]).mkdir(parents=True, exist_ok=True)

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {}
        for cfg in todo:
            cmd = _make_cmd(cfg)
            fut = executor.submit(_run_training_subprocess, cmd, str(ROOT), TIMEOUT_S)
            futures[fut] = cfg

        done_count = len(results)
        total_count = len(run_configs)
        for fut in as_completed(futures):
            cfg = futures[fut]
            done_count += 1
            rc, stdout, stderr = fut.result()
            out_dir = Path(cfg["out_dir"])
            if rc != 0:
                print(f"[{done_count}/{total_count}] FAIL {cfg['model']} {cfg['size']}")
                print(f"  stderr: {tail(stderr, 5)}")
                continue
            log = load_loss(out_dir)
            val = final_metric(log, "val")
            print(f"[{done_count}/{total_count}] {cfg['model']} {cfg['size']} val={val:.4f}")
            results.append((cfg, log))

    elapsed = time.time() - t0
    print(f"[{label}] Done in {elapsed:.0f}s")
    return results


def load_isoflop_c3_rows():
    rows = []
    for model in ISOFLOP_NONBLOCK_MODELS:
        for size in ISOFLOP_NONBLOCK_SIZES:
            run_dir = DATA_ROOT / "isoflop_c3" / model / size
            if not already_done(run_dir):
                continue
            rows.append(
                {
                    "model": model,
                    "size": size,
                    "n_params": MODEL_SIZES[size][2],
                    "run_dir": run_dir,
                    "log": load_loss(run_dir),
                }
            )
    for model in ISOFLOP_BLOCK_MODELS:
        model_tag = model_dir_name(model, BLOCK_LEN)
        for size in SIZE_ORDER:
            run_dir = DATA_ROOT / "isoflop_c3" / model_tag / size
            if not already_done(run_dir):
                continue
            rows.append(
                {
                    "model": model,
                    "size": size,
                    "n_params": MODEL_SIZES[size][2],
                    "run_dir": run_dir,
                    "log": load_loss(run_dir),
                }
            )
    return rows


def load_curriculum_summary(dataset_name: str, sizes, p_values):
    rows = []
    for size in sizes:
        for p_value in p_values:
            run_dir = curriculum_run_dir(dataset_name, size, p_value)
            if not already_done(run_dir):
                continue
            rows.append(
                {
                    "size": size,
                    "p_value": p_value,
                    "run_dir": run_dir,
                    "log": load_loss(run_dir),
                }
            )
    return rows


def save_figure(fig, filename: str) -> Path:
    path = FIG_ROOT / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"Saved {path}")
    return path


print(f"Repo root: {ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Figures:   {FIG_ROOT}")

In [4]:
# Cell 1: IsoFLOP C3 data generation (sequential, skip-if-done)

isoflop_runs = []

for model in ISOFLOP_NONBLOCK_MODELS:
    for size in ISOFLOP_NONBLOCK_SIZES:
        steps = compute_isoflop_steps(ISOFLOP_BUDGET, model, size)
        if steps is None:
            continue
        isoflop_runs.append(
            {
                "model": model,
                "size": size,
                "steps": steps,
                "block_len": None,
                "out_dir": DATA_ROOT / "isoflop_c3" / model / size,
            }
        )

for model in ISOFLOP_BLOCK_MODELS:
    for size in SIZE_ORDER:
        steps = compute_isoflop_steps(ISOFLOP_BUDGET, model, size)
        if steps is None:
            continue
        isoflop_runs.append(
            {
                "model": model,
                "size": size,
                "steps": steps,
                "block_len": BLOCK_LEN,
                "out_dir": DATA_ROOT / "isoflop_c3" / model_dir_name(model, BLOCK_LEN) / size,
            }
        )

assert len(isoflop_runs) == 31, f"Expected 31 C3 IsoFLOP runs, found {len(isoflop_runs)}"
print(f"Planned {len(isoflop_runs)} IsoFLOP C3 runs")

isoflop_results = []
for index, run in enumerate(isoflop_runs, start=1):
    print(
        f"[{index:02d}/{len(isoflop_runs)}] {run['model']} {run['size']} "
        f"steps={run['steps']} block_len={run['block_len']}"
    )
    log = run_training(
        run["model"],
        run["size"],
        run["out_dir"],
        max_iters=run["steps"],
        block_len=run["block_len"],
        lr=get_optimal_lr(run["model"], run["size"]),
        dropout=dropout_for_model(run["model"]),
        eval_interval=isoflop_eval_interval(run["steps"]),
        eval_iters=EVAL_ITERS,
        warmup_iters=isoflop_warmup_iters(run["steps"]),
        save_interval=0,
        gpt2_eval_interval=0,
        gpt2_eval_samples=0,
        num_final_samples=0,
        sample_interval=0,
    )
    row = dict(run)
    row["train_ce"] = final_metric(log, "train")
    row["val_ce"] = final_metric(log, "val")
    isoflop_results.append(row)
    print(f"    final train={row['train_ce']:.4f} | val={row['val_ce']:.4f}")

print(f"\nCompleted {len(isoflop_results)}/{len(isoflop_runs)} IsoFLOP C3 runs")
for row in sorted(isoflop_results, key=lambda item: (item["model"], MODEL_SIZES[item["size"]][2])):
    print(
        f"  {row['model']:<22} {row['size']:<4} "
        f"val={row['val_ce']:.4f} train={row['train_ce']:.4f}"
    )


Planned 31 IsoFLOP C3 runs
[01/31] remasked 0.3M steps=8845 block_len=None
[run] remasked 0.3M -> /workspace/shakespeare-dLM/data/isoflop_c3/remasked/0.3M
    final train=1.9225 | val=2.0472
[02/31] remasked 0.5M steps=5704 block_len=None
[run] remasked 0.5M -> /workspace/shakespeare-dLM/data/isoflop_c3/remasked/0.5M
    final train=1.8867 | val=2.0244
[03/31] remasked 1M steps=3045 block_len=None
[run] remasked 1M -> /workspace/shakespeare-dLM/data/isoflop_c3/remasked/1M
    final train=1.8436 | val=1.9814
[04/31] remasked 2M steps=1565 block_len=None
[run] remasked 2M -> /workspace/shakespeare-dLM/data/isoflop_c3/remasked/2M
    final train=1.8849 | val=2.0108
[05/31] remasked 3M steps=959 block_len=None
[run] remasked 3M -> /workspace/shakespeare-dLM/data/isoflop_c3/remasked/3M
    final train=1.9802 | val=2.0736
[06/31] mdlm 0.3M steps=8845 block_len=None
[run] mdlm 0.3M -> /workspace/shakespeare-dLM/data/isoflop_c3/mdlm/0.3M
    final train=1.9205 | val=2.0488
[07/31] mdlm 0.5M st

In [ ]:
curriculum_1200_runs = [
    (size, p_value)
    for size in CURRICULUM_1200_SIZES
    for p_value in CURRICULUM_1200_P_VALUES
]
n_expected = len(CURRICULUM_1200_SIZES) * len(CURRICULUM_1200_P_VALUES)
assert len(curriculum_1200_runs) == n_expected, f"Expected {n_expected}, got {len(curriculum_1200_runs)}"
print(f"Planned {len(curriculum_1200_runs)} curriculum 1200-step runs")

curriculum_1200_results = []
for index, (size, p_value) in enumerate(curriculum_1200_runs, start=1):
    run_dir = curriculum_run_dir(CURRICULUM_1200_DIR, size, p_value)
    ar_steps, bd_steps = curriculum_steps(CURRICULUM_1200_TOTAL_STEPS, p_value)
    print(f"[{index:02d}/{len(curriculum_1200_runs)}] size={size} p_ar={format_p_value(p_value)}")

    resume_from = None
    if ar_steps > 0:
        phase1_dir = run_dir / "phase1_ar"
        run_training(
            "ar",
            size,
            phase1_dir,
            max_iters=ar_steps,
            lr=get_optimal_lr("ar", size),
            dropout=dropout_for_model("ar"),
            eval_interval=0,
            eval_iters=EVAL_ITERS,
            warmup_iters=ar_warmup_iters(ar_steps),
            save_interval=0,
            gpt2_eval_interval=0,
            gpt2_eval_samples=0,
            num_final_samples=0,
            sample_interval=0,
        )
        resume_from = phase1_dir / "ckpt.pt"

    phase2_log = run_training(
        "block_mdlm",
        size,
        run_dir,
        max_iters=bd_steps,
        block_len=BLOCK_LEN,
        lr=get_optimal_lr("block_mdlm", size),
        dropout=dropout_for_model("block_mdlm"),
        eval_interval=0,
        eval_iters=EVAL_ITERS,
        warmup_iters=bd_warmup_iters(bd_steps),
        resume_from=resume_from,
        save_interval=0,
        gpt2_eval_interval=0,
        gpt2_eval_samples=0,
        num_final_samples=0,
        sample_interval=0,
    )

    row = {
        "size": size,
        "p_value": p_value,
        "ar_steps": ar_steps,
        "bd_steps": bd_steps,
        "train_ce": final_metric(phase2_log, "train"),
        "val_ce": final_metric(phase2_log, "val"),
    }
    curriculum_1200_results.append(row)
    print(f"    final train={row['train_ce']:.4f} | val={row['val_ce']:.4f}")

print("\nFinal 1200-step curriculum summary")
for row in sorted(curriculum_1200_results, key=lambda item: (item["size"], item["p_value"])):
    print(
        f"size={row['size']:<4} p_ar={format_p_value(row['p_value'])} "
        f"ar_steps={row['ar_steps']:<4} bd_steps={row['bd_steps']:<4} val={row['val_ce']:.4f}"
    )

In [6]:
curriculum_4000_runs = [
    (size, p_value)
    for size in CURRICULUM_4000_SIZES
    for p_value in CURRICULUM_4000_P_VALUES
]
assert len(curriculum_4000_runs) == 6
print(f"Planned {len(curriculum_4000_runs)} curriculum 4000-step runs")

curriculum_4000_results = []
for index, (size, p_value) in enumerate(curriculum_4000_runs, start=1):
    run_dir = curriculum_run_dir(CURRICULUM_4000_DIR, size, p_value)
    ar_steps, bd_steps = curriculum_steps(CURRICULUM_4000_TOTAL_STEPS, p_value)
    print(f"[{index:02d}/{len(curriculum_4000_runs)}] size={size} p_ar={format_p_value(p_value)}")

    resume_from = None
    if ar_steps > 0:
        phase1_dir = run_dir / "phase1_ar"
        run_training(
            "ar",
            size,
            phase1_dir,
            max_iters=ar_steps,
            lr=get_optimal_lr("ar", size),
            dropout=dropout_for_model("ar"),
            eval_interval=CURRICULUM_4000_EVAL_INTERVAL,
            eval_iters=EVAL_ITERS,
            warmup_iters=ar_warmup_iters(ar_steps),
            save_interval=0,
            gpt2_eval_interval=0,
            gpt2_eval_samples=0,
            num_final_samples=0,
            sample_interval=0,
        )
        resume_from = phase1_dir / "ckpt.pt"

    phase2_log = run_training(
        "block_mdlm",
        size,
        run_dir,
        max_iters=bd_steps,
        block_len=BLOCK_LEN,
        lr=get_optimal_lr("block_mdlm", size),
        dropout=dropout_for_model("block_mdlm"),
        eval_interval=CURRICULUM_4000_EVAL_INTERVAL,
        eval_iters=EVAL_ITERS,
        warmup_iters=bd_warmup_iters(bd_steps),
        resume_from=resume_from,
        save_interval=0,
        gpt2_eval_interval=0,
        gpt2_eval_samples=0,
        num_final_samples=0,
        sample_interval=0,
    )

    row = {
        "size": size,
        "p_value": p_value,
        "ar_steps": ar_steps,
        "bd_steps": bd_steps,
        "train_ce": final_metric(phase2_log, "train"),
        "val_ce": final_metric(phase2_log, "val"),
    }
    curriculum_4000_results.append(row)
    print(f"    final train={row['train_ce']:.4f} | val={row['val_ce']:.4f}")

print("\nFinal 4000-step curriculum summary")
for row in sorted(curriculum_4000_results, key=lambda item: (item["size"], item["p_value"])):
    print(
        f"size={row['size']:<4} p_ar={format_p_value(row['p_value'])} "
        f"ar_steps={row['ar_steps']:<4} bd_steps={row['bd_steps']:<4} val={row['val_ce']:.4f}"
    )


Planned 6 curriculum 4000-step runs
[01/6] size=0.3M p_ar=0.0
[run] block_mdlm 0.3M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.0_0.3M
    final train=1.6759 | val=1.8604
[02/6] size=0.3M p_ar=0.3
[run] ar 0.3M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.3_0.3M/phase1_ar
[run] block_mdlm 0.3M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.3_0.3M
    final train=1.6283 | val=1.8119
[03/6] size=0.3M p_ar=0.5
[run] ar 0.3M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.5_0.3M/phase1_ar
[run] block_mdlm 0.3M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.5_0.3M
    final train=1.6441 | val=1.8295
[04/6] size=1M p_ar=0.0
[run] block_mdlm 1M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.0_1M
    final train=1.5037 | val=1.7329
[05/6] size=1M p_ar=0.3
[run] ar 1M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.3_1M/phase1_ar
[run] block_mdlm 1M -> /workspace/shakespeare-dLM/data/curriculum_4000/p0.3_1M
    final train=1.4417 | val=1.6903


In [ ]:
rows = load_isoflop_c3_rows()
if len(rows) != 31:
    raise RuntimeError(f"Expected 31 C3 IsoFLOP runs, found {len(rows)}. Run Cell 1 first.")

fig, ax = plt.subplots(figsize=(8.5, 5.5))

# Since remasked≡MDLM and block_remasked≡block_mdlm during training,
# plot only one representative from each pair.
fig1_models = ["remasked", "block_remasked", "block_edit_one_pass", "block_edit_two_pass"]

for model in fig1_models:
    points = sorted(
        [row for row in rows if row["model"] == model],
        key=lambda row: row["n_params"],
    )
    xs = [row["n_params"] for row in points]
    ys = [final_metric(row["log"], "val") for row in points]
    linestyle = "--" if model in ISOFLOP_NONBLOCK_MODELS else "-"
    ax.plot(
        xs,
        ys,
        marker="o",
        linewidth=2.2,
        markersize=6,
        linestyle=linestyle,
        color=MODEL_COLORS[model],
        label=MODEL_LABELS[model],
    )

ax.set_xscale("log")
xticks = [MODEL_SIZES[size][2] for size in SIZE_ORDER]
ax.set_xticks(xticks)
ax.set_xticklabels(SIZE_ORDER)
ax.set_xlabel("model size")
ax.set_ylabel("final validation CE")
ax.set_title("Figure 1. IsoFLOP C3: block diffusion vs standard diffusion")
ax.legend(ncol=2, frameon=False)

save_figure(fig, "figure1_isoflop_block_vs_nonblock.png")
plt.show()

In [ ]:
rows = load_isoflop_c3_rows()
if len(rows) != 31:
    raise RuntimeError(f"Expected 31 C3 IsoFLOP runs, found {len(rows)}. Run Cell 1 first.")

fig, ax = plt.subplots(figsize=(8.5, 5.5))

# Since block_remasked≡block_mdlm during training, plot only one representative.
fig2_models = ["block_remasked", "block_edit_one_pass", "block_edit_two_pass"]
block_rows = [row for row in rows if row["model"] in ISOFLOP_BLOCK_MODELS]

for model in fig2_models:
    points = sorted(
        [row for row in block_rows if row["model"] == model],
        key=lambda row: row["n_params"],
    )
    xs = [row["n_params"] for row in points]
    ys = [final_metric(row["log"], "val") for row in points]
    ax.plot(
        xs,
        ys,
        marker="o",
        linewidth=2.4,
        markersize=6,
        color=MODEL_COLORS[model],
        label=MODEL_LABELS[model],
    )
    best = min(points, key=lambda row: final_metric(row["log"], "val"))
    best_y = final_metric(best["log"], "val")
    ax.scatter(best["n_params"], best_y, marker="*", s=220, color=MODEL_COLORS[model], edgecolor="black", linewidth=0.6, zorder=5)
    ax.annotate(best["size"], (best["n_params"], best_y), xytext=(0, 12), textcoords="offset points", ha="center", fontsize=9)

# Overfitting region: 1M–3M (was 2M–3M)
ax.axvspan(MODEL_SIZES["1M"][2], MODEL_SIZES["3M"][2] * 1.03, color="0.92", zorder=0)
ax.text(MODEL_SIZES["1M"][2] * 1.03, ax.get_ylim()[1] - 0.02, "1M–3M overfitting region", color="0.35", fontsize=9)
ax.set_xscale("log")
xticks = [MODEL_SIZES[size][2] for size in SIZE_ORDER]
ax.set_xticks(xticks)
ax.set_xticklabels(SIZE_ORDER)
ax.set_xlabel("model size")
ax.set_ylabel("final validation CE")
ax.set_title("Figure 2. IsoFLOP C3: block model ranking")
ax.legend(frameon=False)

save_figure(fig, "figure2_block_model_ranking.png")
plt.show()

In [ ]:
rows_1200 = load_curriculum_summary(CURRICULUM_1200_DIR, CURRICULUM_1200_SIZES, CURRICULUM_1200_P_VALUES)
rows_4000 = load_curriculum_summary(CURRICULUM_4000_DIR, CURRICULUM_4000_SIZES, CURRICULUM_4000_P_VALUES)

n_expected_1200 = len(CURRICULUM_1200_SIZES) * len(CURRICULUM_1200_P_VALUES)
n_expected_4000 = len(CURRICULUM_4000_SIZES) * len(CURRICULUM_4000_P_VALUES)
if len(rows_1200) != n_expected_1200:
    raise RuntimeError(f"Expected {n_expected_1200} curriculum_1200 runs, found {len(rows_1200)}. Run Cell 2 first.")
if len(rows_4000) != n_expected_4000:
    raise RuntimeError(f"Expected {n_expected_4000} curriculum_4000 runs, found {len(rows_4000)}. Run Cell 3 first.")

# Both panels now use the same sizes (0.3M, 1M)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=False)
left, right = axes

for size in CURRICULUM_1200_SIZES:
    points = sorted(
        [row for row in rows_1200 if row["size"] == size],
        key=lambda row: row["p_value"],
    )
    xs = [row["p_value"] for row in points]
    ys = [final_metric(row["log"], "val") for row in points]
    left.plot(xs, ys, marker="o", linewidth=2.2, markersize=6, color=SIZE_COLORS[size], label=size)
    best_index = min(range(len(points)), key=lambda i: ys[i])
    left.scatter(xs[best_index], ys[best_index], marker="*", s=180, color=SIZE_COLORS[size], edgecolor="black", linewidth=0.6, zorder=5)

left.axvline(0.3, color="0.45", linestyle="--", linewidth=1)
left.set_xticks(CURRICULUM_1200_P_VALUES)
left.set_xlabel("AR pretrain ratio p_ar")
left.set_ylabel("final validation CE")
left.set_title("1200 total steps")
left.legend(title="block MDLM size", frameon=False)

for size in CURRICULUM_4000_SIZES:
    points = sorted(
        [row for row in rows_4000 if row["size"] == size],
        key=lambda row: row["p_value"],
    )
    xs = [row["p_value"] for row in points]
    ys = [final_metric(row["log"], "val") for row in points]
    right.plot(xs, ys, marker="o", linewidth=2.2, markersize=6, color=SIZE_COLORS[size], label=size)
    best_index = min(range(len(points)), key=lambda i: ys[i])
    right.scatter(xs[best_index], ys[best_index], marker="*", s=180, color=SIZE_COLORS[size], edgecolor="black", linewidth=0.6, zorder=5)

right.axvline(0.3, color="0.45", linestyle="--", linewidth=1)
right.set_xticks(CURRICULUM_4000_P_VALUES)
right.set_xlabel("AR pretrain ratio p_ar")
right.set_ylabel("final validation CE")
right.set_title("4000 total steps")
right.legend(title="block MDLM size", frameon=False)

fig.suptitle("Figure 3. Curriculum sweet spot stays at p_ar = 0.3", y=1.02)
fig.tight_layout()
save_figure(fig, "figure3_curriculum_sweet_spot.png")
plt.show()

In [ ]:
rows_4000 = load_curriculum_summary(CURRICULUM_4000_DIR, CURRICULUM_4000_SIZES, CURRICULUM_4000_P_VALUES)
n_expected_4000 = len(CURRICULUM_4000_SIZES) * len(CURRICULUM_4000_P_VALUES)
if len(rows_4000) != n_expected_4000:
    raise RuntimeError(f"Expected {n_expected_4000} curriculum_4000 runs, found {len(rows_4000)}. Run Cell 3 first.")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=True)
transition_steps, _ = curriculum_steps(CURRICULUM_4000_TOTAL_STEPS, 0.3)

for ax, size in zip(axes, CURRICULUM_4000_SIZES):
    baseline_dir = curriculum_run_dir(CURRICULUM_4000_DIR, size, 0.0)
    curriculum_dir = curriculum_run_dir(CURRICULUM_4000_DIR, size, 0.3)
    phase1_dir = curriculum_dir / "phase1_ar"

    baseline_log = load_loss(baseline_dir)
    phase1_log = load_loss(phase1_dir)
    phase2_log = load_loss(curriculum_dir)

    x_base, y_base = curve_points(baseline_log, split="val")
    x_ar, y_ar = curve_points(phase1_log, split="val")
    x_bd, y_bd = curve_points(phase2_log, split="val", offset=transition_steps)

    ax.plot(x_base, y_base, color="0.65", linewidth=2.0, label="p=0.0 pure BD")
    ax.plot(x_ar, y_ar, color="#4c72b0", linewidth=2.2, label="p=0.3 AR phase")
    ax.plot(x_bd, y_bd, color="#55a868", linewidth=2.2, label="p=0.3 BD phase")
    ax.axvline(transition_steps, color="0.35", linestyle="--", linewidth=1)
    ax.set_title(f"{size} block MDLM")
    ax.set_xlabel("training step")
    ax.set_ylabel("val CE")
    ax.set_xlim(0, CURRICULUM_4000_TOTAL_STEPS)

axes[0].legend(frameon=False)
fig.suptitle("Figure 4. Validation loss for p_ar = 0.3 curriculum vs pure block diffusion", y=1.02)
fig.tight_layout()
save_figure(fig, "figure4_curriculum_loss_curves.png")
plt.show()